In [0]:
from pyspark.sql.functions import current_timestamp, col,current_date
import re

In [0]:
display(dbutils.fs.ls("/Volumes/sdp_investment/bronze/raw_mutual_fund_data/mutual_fund_02_10_2026.csv"))

In [0]:
source_path = "/Volumes/investment_pyspark/bronze/landing/mutual_fund_holding/"
schema_path = "/Volumes/investment_pyspark/bronze/landing/mutual_fund_holding/mutual_fund_schemas/"
checkpoint_path = ("/Volumes/investment_pyspark/bronze/landing/mutual_fund_holding/mutual_fund_checkpoints/mutual_fund_holdings")

def clean_column_name(column_name):
    column_name = column_name.strip()
    column_name = re.sub(r'[^a-zA-Z0-9_]', '_', column_name)
    column_name = re.sub(r'_+', '_', column_name)
    return column_name.strip('_')

In [0]:
df_bronze = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("pathGlobFilter", "*.csv")
    .option("header","true")
    .load(source_path)
)
df_cleaned = df_bronze.toDF(*[clean_column_name(c) for c in df_bronze.columns])
df_transformed = (df_cleaned
                  .withColumn("_ingestion_timestamp", current_timestamp())
                  .withColumn("_source_file", col("_metadata.file_path"))
                  .withColumn("_snapshot_date", current_date())
                  )
query = (df_transformed.writeStream
         .format("delta")
         .option("checkpointLocation", checkpoint_path)
         .outputMode("append")
         .trigger(availableNow=True)
         .toTable("investment_pyspark.bronze.mutual_fund_holdings_raw"))
query.awaitTermination()
print(f"Stream completed. Processed {query.lastProgress}")

In [0]:
display(df_transformed.columns)